This notebook extends the ["Checkpointers"](./Checkpointers.ipynb) example.

In [ ]:
# Install LangChain and the OpenAI integration.
# This notebook also uses LangGraph's Store feature (included in langgraph, installed via langchain).
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain.agents import AgentState         # Built-in LangGraph state with a 'messages' list
from langchain.tools import ToolRuntime, tool   # ToolRuntime: gives tools access to context and store at runtime
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable, RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver   # Stores per-thread conversation checkpoints in RAM
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode
from langgraph.store.base import BaseStore              # Abstract base class for all LangGraph stores
from langgraph.store.memory import InMemoryStore        # Key-value store shared ACROSS all threads (global scope)
from pathlib import Path
from pydantic import SecretStr
from typing import List, TypedDict

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper: pretty-prints every message in a conversation
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

# Helper: renders the compiled graph as a PNG and shows it inline in Jupyter
def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

# Helper: lists all checkpoints saved for a given thread config
def explore_checkpoints(checkpointer: BaseCheckpointSaver, config: RunnableConfig):
    checkpoints = list(checkpointer.list(config))
    print(f"There are {len(checkpoints)} checkpoints in total:")
    for checkpoint in reversed(checkpoints):
        print(checkpoint)

# Helper: prints the graph state at each execution step
def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

# Helper: prints everything currently stored in the cross-thread store.
# Iterates over all namespaces and prints each stored key-value item.
def explore_store(store: BaseStore):
    for ns in store.list_namespaces():
        print(ns)  # e.g. ('users', 'troeff_1', 'recent_activity')

        search_items = store.search(ns, limit=100)
        for item in search_items:
            print(f"{item.key}: {item.value}")

        print()

In [ ]:
# NEW CONCEPT: Context
# A LangGraph 'Context' is a TypedDict passed to the graph at invocation time.
# Unlike the State (which lives inside the graph and changes as nodes run),
# the Context is READ-ONLY metadata provided by the CALLER — it doesn't flow through the graph.
# Typical uses: user identity, tenant ID, feature flags, request metadata.

# This is the graph state — the mutable data shared across all nodes
class CustomAgentState(AgentState):
    model_calls: int  # Tracks how many times the model node has been invoked

# This is the per-invocation context — immutable metadata provided when calling graph.invoke()
class CustomAgentContext(TypedDict):
    user_id: str  # Identifies which user is making the request (passed to tools via ToolRuntime)

In [ ]:
# NEW CONCEPT: ToolRuntime
# When a tool needs access to the context or the store, it declares a special
# `runtime: ToolRuntime[TContext]` parameter. LangGraph injects this automatically —
# you never pass it manually; the ToolNode handles it behind the scenes.
#
# NOTE: To access `context` and `store`, we use `ToolRuntime[TContext]` for tools and `Runtime[TContext]` for regular graph nodes.

@tool
def weather(city: str, runtime: ToolRuntime[CustomAgentContext]) -> str:
    """Return a (fake) current-weather report for a city."""
    # runtime.context contains the per-invocation metadata (e.g. user_id)
    print(f"Weather tool call executing for user \"{runtime.context['user_id']}\".")

    # Store the weather request in the cross-thread Store under a user-specific namespace.
    # Namespace structure: ("users", <user_id>, "recent_activity")
    # This data persists ACROSS conversation threads — unlike state/checkpoints which are per-thread.
    namespace = ("users", runtime.context['user_id'], "recent_activity")
    runtime.store.put(namespace, "weather_request", { "city": city })

    # Simulated weather data — in a real app this would call an external weather API
    data = {
        "sofia": "Sofia: 18 C, partly cloudy",
        "london": "London: 11 C, rainy",
        "tokyo": "Tokyo: 22 C, sunny",
    }
    return data.get(city.lower(), f"No data for {city}.")

@tool
def search(query: str, runtime: ToolRuntime[CustomAgentContext]) -> str:
    """Look up a term in the built-in mini-encyclopedia."""
    print(f"Search tool call executing for user \"{runtime.context['user_id']}\".")

    # Store the search query for this user in the same namespace
    namespace = ("users", runtime.context['user_id'], "recent_activity")
    runtime.store.put(namespace, "search_request", { "query": query })

    # Simulated knowledge base
    data = {
        "langgraph": "LangGraph is a library for building stateful, cyclic LLM apps.",
        "react": "ReAct is a prompting pattern: Reason then Act, in a loop.",
        "dag": "A DAG is a directed acyclic graph — no cycles allowed.",
    }
    return data.get(query.lower(), "(nothing found)")


SYSTEM_PROMPT = "You are a concise assistant. Use tools when useful."
TOOLS = [weather, search]

In [ ]:
# Bind the tools to the model (same pattern as AI Agent notebooks)
model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(TOOLS)

# Lifecycle hook nodes — same as AI Agent Part 2, added for observability/logging
def on_start(state: CustomAgentState):
    # Fires once at the very beginning of each graph invocation
    print("Our graph execution started!")

def on_end(state: CustomAgentState):
    # Fires once just before the graph exits
    print("Our graph execution just ended!")

def before_model_node(state: CustomAgentState):
    # Runs before each LLM call — increments and logs the model call counter
    model_call_id = state.get('model_calls', 0) + 1
    print('=' * 20)
    print(f"Starting model call #{model_call_id}...")
    return { "model_calls": model_call_id }

def after_model_node(state: CustomAgentState):
    # Runs after each LLM call — logs the completed call number
    model_call_id = state.get('model_calls', 0)
    print(f"Finished model call #{model_call_id}.")
    print('=' * 20)

def model_node(state: CustomAgentState):
    # The actual GPT call — prepends the system prompt to the full conversation history
    response = model.invoke([SystemMessage(SYSTEM_PROMPT), *state["messages"]])
    return { "messages": [response] }

In [ ]:
# Routing function — checks if the model's last message contains pending tool calls.
# Returns True  => go to the "tools" node to execute them
# Returns False => the model is done, go to "on_end" and then EXIT
def has_pending_tool_calls(state: CustomAgentState) -> bool:
    messages = state.get("messages", [])
    if not messages:
        return False  # Nothing in the conversation yet

    last_message = messages[-1]
    # AIMessage.tool_calls is non-empty when the model requested one or more tool calls
    return isinstance(last_message, AIMessage) and last_message.tool_calls

In [ ]:
# KEY CONCEPT: Checkpointer vs. Store — what's the difference?
#
# InMemorySaver (checkpointer):
#   - Stores state snapshots (messages, model_calls counter, etc.)
#   - Scoped to a single THREAD (conversation session)
#   - Used to enable multi-turn conversations and state replay
#
# InMemoryStore (store):
#   - A global key-value store shared ACROSS all threads
#   - Namespaced by arbitrary tuples (e.g. ('users', 'user_id', 'recent_activity'))
#   - Used to persist data that should outlive a single conversation (e.g. user preferences, activity logs)
#
in_memory_checkpointer = InMemorySaver()
in_memory_store = InMemoryStore()

In [ ]:
# Build the graph — same structure as AI Agent Part 2, with two new additions:
#
# 1. StateGraph now takes a SECOND argument: CustomAgentContext
#    This registers the context type so LangGraph knows how to inject it into tools via ToolRuntime.
#
# 2. graph.compile() receives BOTH a checkpointer AND a store:
#    - checkpointer: enables per-thread state persistence (conversation memory)
#    - store:        enables cross-thread data persistence (shared global storage)
graph_builder = StateGraph(CustomAgentState, CustomAgentContext)
graph_builder.add_node("on_start", on_start)
graph_builder.add_node("on_end", on_end)
graph_builder.add_node("before_model", before_model_node)
graph_builder.add_node("after_model", after_model_node)
graph_builder.add_node("model", model_node)
graph_builder.add_node("tools", ToolNode(TOOLS))

# Wire the execution order (same as AI Agent Part 2)
graph_builder.add_edge(START, "on_start")
graph_builder.add_edge("on_start", "before_model")
graph_builder.add_edge("before_model", "model")
graph_builder.add_edge("model", "after_model")
graph_builder.add_conditional_edges("after_model", lambda x: "tools" if has_pending_tool_calls(x) else "on_end", ["tools", "on_end"])
graph_builder.add_edge("tools", "before_model")
graph_builder.add_edge("on_end", END)

# Compile with BOTH persistence layers:
#   checkpointer=in_memory_checkpointer -> per-thread state snapshots
#   store=in_memory_store               -> global cross-thread key-value storage
graph = graph_builder.compile(checkpointer=in_memory_checkpointer, store=in_memory_store)

In [ ]:
# Visualize the graph — the structure is identical to AI Agent Part 2; the store is invisible in the diagram
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Invoke the graph with:
#   - config:   the thread_id (same as Checkpointers.ipynb — scopes conversation memory)
#   - context:  the user_id (new — passed as read-only metadata to tools via ToolRuntime)
#
# When the tools run, they will write to the store under the key:
#   ('users', 'troeff_1', 'recent_activity')
thread1_config = { "configurable": { "thread_id": "thread_1" } }
thread1_context = { "user_id": "troeff_1" }
thread1_result = graph.invoke(
    input={
        "messages": [HumanMessage("What's the weather in Tokyo and what is LangGraph, briefly?")]
    },
    config=thread1_config,
    context=thread1_context   # This is the NEW parameter — provides the context to the graph
)

In [ ]:
# Print the full conversation: user question -> tool calls -> tool results -> final AI answer
print_conversation(thread1_result["messages"])

In [ ]:
# Walk through the per-thread state history (checkpointer data).
# This shows how state evolved step by step within thread_1 — same as Checkpointers.ipynb.
explore_state_history(graph, thread1_config)

In [ ]:
# Inspect the global Store — this is data written by the tools ACROSS all threads.
# You should see the 'weather_request' and 'search_request' entries written by the tools,
# stored under the namespace ('users', 'troeff_1', 'recent_activity').
# Unlike checkpoints, this data is NOT tied to thread_1 — it would be accessible from any thread.
explore_store(in_memory_store)